In [ ]:
import requests
import pandas as pd
from tqdm import tqdm

API_KEY = "f27f2545bf97f91cfa9231bacd919c0fa477645482ad8dd57aa5ba87aab75623a87614a78937c9977e78dfa3c7976575dc984bdb6c5e3e60ed4f674058251ac6"

API_URL = "https://www.sima-land.ru/api/v3/item/"
PER_PAGE = 100
CATEGORY_IDS = [11091, 3003, 363, 39547]
all_items = []

for category_id in CATEGORY_IDS:
    print(f"Категория {category_id}")
    page = 1
    with tqdm(desc=f"Категория {category_id}", unit="стр", ncols=80) as pbar:
        while True:
            params = {
                "per-page": PER_PAGE,
                "page": page,
                "category_id": category_id,
            }
            headers = {
                "Accept": "application/json",
                "x-api-key": API_KEY,
            }
            r = requests.get(API_URL, params=params, headers=headers)
            data = r.json()

            items = data if isinstance(data, list) else data.get("items", [])
            if not items:
                break

            for item in items:
                item["source_category"] = category_id
            all_items.extend(items)

            pbar.update(len(items))
            page += 1

print(f"\n Всего товаров собрано: {len(all_items)}")

# Сохраняем всё в CSV (UTF-8 с BOM — для Excel)
df = pd.DataFrame(all_items)
df.to_csv("items_all.csv", index=False, encoding="utf-8-sig")




Категория 11091


Категория 11091: 10100стр [09:23, 17.91стр/s]


Категория 3003


Категория 3003: 10100стр [06:15, 26.92стр/s]


Категория 363


Категория 363: 10100стр [09:34, 17.57стр/s]


Категория 39547


Категория 39547: 8742стр [05:45, 25.28стр/s]



 Всего товаров собрано: 39042


In [ ]:
import requests
import pandas as pd

API_KEY = "f27f2545bf97f91cfa9231bacd919c0fa477645482ad8dd57aa5ba87aab75623a87614a78937c9977e78dfa3c7976575dc984bdb6c5e3e60ed4f674058251ac6"
API_URL = "https://www.sima-land.ru/api/v3/category/"
PER_PAGE = 100

all_categories = []
last_id = 0
while True:
    params = {
        "per-page": PER_PAGE,
        "id-greater-than": last_id,
        "level": 1
    }
    headers = {
        "Accept": "application/json",
        "x-api-key": API_KEY
    }

    r = requests.get(API_URL, params=params, headers=headers)
    data = r.json()

    items = data.get("items", [])

    for item in items:
        all_categories.append({
            "id": item.get("id"),
            "name": item.get("name"),
            "is_not_empty": item.get("is_not_empty"),
        })
        if "id" in item:
            last_id = max(last_id, int(item["id"]))

    if len(items) < PER_PAGE:
        break

print(f"Всего категорий: {len(all_categories)}")

df = pd.DataFrame(all_categories)
df.to_csv("top_categories.csv", index=False, encoding="utf-8-sig")


Всего категорий: 35


Парсим сначала категории, которые есть на сималенде, потом парсим товары категорий.